# Lab 01.2 — Il paradigma MapReduce con Spark (RDD)

## Introduzione
**MapReduce** scompone un calcolo in tre fasi:

1. **Map** — ogni record viene trasformato in coppie `(chiave, valore)`; eseguito in parallelo su ogni partizione.
2. **Shuffle** — le coppie vengono ridistribuite in rete in modo che tutte quelle con la stessa chiave finiscano nella stessa partizione.
3. **Reduce** — i valori di ogni chiave vengono aggregati.

Spark implementa lo stesso modello con le operazioni sugli **RDD** (Resilient Distributed Dataset), ma mantiene i risultati intermedi in memoria e ottimizza l'intera catena come un DAG.

### Obiettivi
1. Inizializzare la `SparkSession` collegata al cluster.
2. Leggere un file di testo in modo distribuito (partizioni).
3. Implementare il **Word Count** fase per fase (map, shuffle, reduce) ed esaminarne il lineage.
4. Osservare come lo shuffle distribuisce le chiavi tra le partizioni (`glom`).
5. Confrontare `reduceByKey` e `groupByKey`.
6. Costruire un **indice invertito** come secondo esempio MapReduce.
7. Riscrivere Word Count e indice invertito con la **API DataFrame** e confrontarli con gli RDD (risultati e tempi).
8. Eseguire il calcolo come **job batch** con `spark-submit`.

## Step 1: Inizializzare la SparkSession
Creiamo la sessione usando il master definito dalla variabile d'ambiente `SPARK_MASTER` (il cluster Standalone del lab); se manca, si ripiega su `local[*]`. Otteniamo anche la `SparkContext` (`sc`), necessaria per le operazioni sugli RDD.

In [ ]:
import os, re, time
from pyspark.sql import SparkSession

# Il master arriva dall'ambiente (docker-compose); in sua assenza si usa la modalità locale
spark = (SparkSession.builder
         .appName("Lab01-MapReduce")
         .master(os.environ.get("SPARK_MASTER", "local[*]"))
         .getOrCreate())
sc = spark.sparkContext
print("Master:", sc.master)

## Step 2: Lettura distribuita del testo
Il file viene diviso in **partizioni**: ciascuna sarà elaborata da un task.
La cartella `datasets/` è montata in tutti i container (driver ed executor), come se fosse un filesystem condiviso: per questo si usa un percorso `file://` identico su ogni nodo.
Con `minPartitions=4` chiediamo almeno 4 partizioni, poi contiamo le righe e ne mostriamo alcune non vuote.

In [ ]:
# Percorso identico su driver ed executor (volume montato in tutti i container)
PATH = "file:///home/jovyan/datasets/alice_in_wonderland.txt"
lines = sc.textFile(PATH, minPartitions=4)

print("Partizioni :", lines.getNumPartitions())
print("Righe      :", lines.count())
# Anteprima: prime 5 righe non vuote
for l in lines.filter(lambda l: l.strip()).take(5):
    print("  |", l)

## Step 3: Word Count, fase per fase
Implementiamo le tre fasi di MapReduce:

| Fase | Operazione | Cosa succede |
|---|---|---|
| Map | `flatMap` + `map` | ogni riga diventa parole e poi coppie `(parola, 1)` |
| Shuffle + Reduce | `reduceByKey` | le coppie con la stessa chiave vengono raggruppate e sommate |
| Azione | `takeOrdered` | restituisce al driver le 15 parole più frequenti |

Il grafico testuale a barre (`#`) rende immediato il confronto tra le frequenze.

In [ ]:
def tokenize(line):
    """Minuscolo, solo lettere: 'Alice,' -> 'alice'."""
    return re.findall(r"[a-z]+", line.lower())

# MAP: riga -> parole -> coppie (parola, 1)
words = lines.flatMap(tokenize)
pairs = words.map(lambda w: (w, 1))
print("Esempio output fase MAP:", pairs.take(6))

# SHUFFLE + REDUCE: somma dei valori con la stessa chiave
counts = pairs.reduceByKey(lambda a, b: a + b)

# Azione: le 15 parole più frequenti (ordinate per conteggio decrescente)
top = counts.takeOrdered(15, key=lambda kv: -kv[1])
for w, c in top:
    print(f"{w:>8} {c:6d}  {'#' * (c // 40)}")

### Step 3b: Lineage e stage
Il **lineage** mostra il confine di *stage* introdotto dallo shuffle (`reduceByKey`).
Nella Spark UI (http://localhost:4040 → *Jobs* → *DAG Visualization*) si vedono due stage: map e reduce.

In [ ]:
# Il rientro nell'indentazione del lineage indica il confine di stage (shuffle)
print(counts.toDebugString().decode())

### Punti chiave (Word Count)
* `flatMap` + `map` realizzano la fase Map; `reduceByKey` realizza Shuffle e Reduce.
* Lo shuffle è il confine tra due stage: lo si vede nel lineage e nella Spark UI.
* Solo l'azione (`takeOrdered`) fa partire il calcolo.

## Step 4: Partizioni, cosa fa ogni task
`glom()` trasforma ogni partizione in una lista: vediamo come le chiavi sono state distribuite dopo lo shuffle (hash partitioning).
Cerchiamo poi in quale partizione si trova la chiave `"alice"`.

In [ ]:
# glom(): una lista per partizione; len() = numero di chiavi in quella partizione
sizes = counts.glom().map(len).collect()
print("Chiavi per partizione dopo lo shuffle:", sizes)
# Ogni chiave va nella partizione hash(chiave) % n_partizioni: tutte le occorrenze
# di "alice" (da qualunque partizione di input) finiscono nella stessa partizione di output
part_alice = counts.mapPartitionsWithIndex(lambda i, it: [i for k, _ in it if k == "alice"]).collect()
print("'alice' si trova nella partizione:", part_alice)

## Step 5: `reduceByKey` vs `groupByKey`
* `groupByKey` sposta in rete **tutte** le coppie e poi somma.
* `reduceByKey` esegue una **pre-aggregazione locale** (il *combiner* di Hadoop MapReduce) prima dello shuffle: molti meno dati trasferiti.

Per un confronto equo mettiamo in cache le coppie (`cache()` + `count()` per materializzarle) e cronometriamo le due strategie, verificando che il risultato sia lo stesso.

In [ ]:
# Riportiamo le coppie in 8 partizioni e le teniamo in memoria per non ricalcolarle
big = pairs.repartition(8).cache()
big.count()  # materializza la cache

# groupByKey: sposta tutte le coppie, poi somma
t0 = time.time(); r1 = big.groupByKey().mapValues(sum).count(); t_group = time.time() - t0
# reduceByKey: pre-aggregazione locale prima dello shuffle
t0 = time.time(); r2 = big.reduceByKey(lambda a, b: a + b).count(); t_reduce = time.time() - t0
print(f"groupByKey : {t_group:.2f}s   reduceByKey: {t_reduce:.2f}s   (stesso risultato: {r1 == r2})")
print("Confronta 'Shuffle Write' dei due job nella scheda Stages della Spark UI.")
big.unpersist()

### Punti chiave (partizioni e shuffle)
* Dopo lo shuffle ogni chiave si trova in una sola partizione (hash partitioning).
* `reduceByKey` riduce i dati trasferiti grazie al combiner: preferirlo a `groupByKey` quando si aggrega.
* La differenza si vede soprattutto in *Shuffle Write* nella Spark UI.

## Step 6: Un secondo esempio MapReduce, l'indice invertito
Per ogni parola, l'elenco dei capitoli in cui compare. È il cuore di un motore di ricerca.
Prima numeriamo i capitoli sul driver (il testo è piccolo), poi applichiamo le tre fasi: **map** (parola, capitolo), **shuffle** (`distinct` + `groupByKey`) e **reduce** (ordinamento dei capitoli).

In [ ]:
# Associamo ogni riga al proprio capitolo (il testo usa le intestazioni "= CHAPTER I =")
text = lines.collect()        # testo piccolo: lo raggruppiamo sul driver per numerare i capitoli
chapter, tagged = 0, []
for l in text:
    if re.match(r"^\W*CHAPTER\s+[IVXLC]+", l):
        chapter += 1
    tagged.append((chapter, l))
chapters = sc.parallelize(tagged, 4)

inverted = (chapters
            .flatMap(lambda cl: [(w, cl[0]) for w in tokenize(cl[1]) if len(w) > 3])  # MAP
            .distinct()
            .groupByKey()                                                               # SHUFFLE
            .mapValues(lambda chs: sorted(chs)))                                        # REDUCE

# Alcune parole di esempio: lookup() restituisce i valori associati a una chiave
for w in ["alice", "rabbit", "queen", "hatter", "turtle"]:
    print(f"{w:>7}: capitoli {inverted.lookup(w)[0]}")

### Punti chiave (indice invertito)
* Lo stesso schema map, shuffle, reduce risolve problemi diversi dal conteggio.
* `distinct` evita duplicati prima del raggruppamento.

## Step 7: Lo stesso calcolo con la API DataFrame
Spark SQL esprime la stessa logica in modo dichiarativo; l'ottimizzatore Catalyst sceglie il piano fisico (che include ancora uno shuffle).
Spezziamo ogni riga in parole con `split` ed `explode`, scartiamo le stringhe vuote e contiamo con `groupBy`; `explain()` mostra il piano fisico.

In [ ]:
from pyspark.sql import functions as F

# riga -> parole (explode) -> conteggio per parola -> ordine decrescente
df = (spark.read.text(PATH)
      .select(F.explode(F.split(F.lower("value"), "[^a-z]+")).alias("word"))
      .where("word != ''")
      .groupBy("word").count()
      .orderBy(F.desc("count")))
df.show(10)
df.explain()   # cerca 'Exchange hashpartitioning' = lo shuffle

### Step 7b: Confronto con gli RDD, risultati e tempi
Il DataFrame dà lo **stesso risultato** degli RDD? E quanto costa in tempo?
Il testo è piccolo e i tempi sarebbero dominati dall'overhead di avvio dei job: lo **ripetiamo 50 volte** (`N`) in entrambe le versioni, così il calcolo pesa davvero.
Ogni versione è eseguita 3 volte e teniamo il tempo migliore, per ridurre il rumore (cache fredda, avvio dei processi Python).
Le differenze principali: gli RDD eseguono `tokenize` in **processi Python** (ogni record viene serializzato avanti e indietro tra JVM e Python), mentre il DataFrame usa funzioni built-in che girano **nella JVM** e sono ottimizzate da Catalyst.

In [ ]:
from functools import reduce

N = 50  # quante volte replichiamo il testo

# RDD: stessa catena dello Step 3, su N copie del testo
def wordcount_rdd():
    return (sc.union([lines] * N)
              .flatMap(tokenize).map(lambda w: (w, 1))
              .reduceByKey(lambda a, b: a + b)
              .collectAsMap())

# DataFrame: stessa logica dello Step 7, su N copie del testo
text_df = spark.read.text(PATH)
big_text_df = reduce(lambda a, b: a.union(b), [text_df] * N)
def wordcount_df():
    rows = (big_text_df
            .select(F.explode(F.split(F.lower("value"), "[^a-z]+")).alias("word"))
            .where("word != ''")
            .groupBy("word").count()
            .collect())
    return {r["word"]: r["count"] for r in rows}

def best_of(fn, n=3):
    tempi, ris = [], None
    for _ in range(n):
        t0 = time.time(); ris = fn(); tempi.append(time.time() - t0)
    return min(tempi), ris

t_rdd, res_rdd = best_of(wordcount_rdd)
t_df,  res_df  = best_of(wordcount_df)
print(f"RDD       : {t_rdd:.2f}s  ({len(res_rdd)} parole distinte)")
print(f"DataFrame : {t_df:.2f}s  ({len(res_df)} parole distinte)")
print(f"Stesso risultato: {res_rdd == res_df}")

### Step 7c: L'indice invertito con i DataFrame
Anche lo Step 6 si può scrivere in modo dichiarativo. Due differenze rispetto agli RDD:
- il **numero di capitolo** non si calcola con un ciclo sul driver, ma con una **funzione finestra**: somma cumulativa delle righe che iniziano con `CHAPTER ...`, nell'ordine delle righe (`monotonically_increasing_id` ne conserva l'ordine nel file);
- `distinct` + `groupByKey` + `sorted` diventano un solo `groupBy("word").agg(collect_set(...))`.

> `Window.orderBy` senza `partitionBy` porta tutte le righe in una sola partizione: va bene per questo testo piccolo, ma non scala su dataset grandi (Spark lo segnala con un avviso).

In [ ]:
from pyspark.sql.window import Window

CAP = r"^\W*CHAPTER\s+[IVXLC]+"   # la stessa regex dello Step 6

inverted_df = (spark.read.text(PATH)
    .withColumn("id", F.monotonically_increasing_id())                       # ordine delle righe
    .withColumn("chapter",
                F.sum(F.col("value").rlike(CAP).cast("int")).over(Window.orderBy("id")))
    .select("chapter", F.explode(F.split(F.lower("value"), "[^a-z]+")).alias("word"))   # MAP
    .where(F.length("word") > 3)
    .groupBy("word")                                                                    # SHUFFLE
    .agg(F.array_sort(F.collect_set("chapter")).alias("chapters")))                     # REDUCE

for w in ["alice", "rabbit", "queen", "hatter", "turtle"]:
    print(f"{w:>7}: capitoli {inverted_df.where(F.col('word') == w).first()['chapters']}")

# Verifica: stesso indice di quello costruito con gli RDD allo Step 6
idx_rdd = dict(inverted.collect())
idx_df = {r["word"]: r["chapters"] for r in inverted_df.collect()}
print("Stesso indice degli RDD:", idx_rdd == idx_df)

### Punti chiave (RDD vs DataFrame)
* Il DataFrame esprime il "cosa", Catalyst decide il "come".
* Nel piano fisico lo shuffle compare come `Exchange hashpartitioning`.
* Stesso risultato, meno codice e di solito meno tempo: le funzioni built-in girano nella JVM, mentre le lambda degli RDD girano in processi Python.
* Con i DataFrame non si sceglie tra `reduceByKey` e `groupByKey`: la pre-aggregazione locale la decide Catalyst.
* Gli RDD restano utili per vedere le fasi del MapReduce una per una e per logiche che non si esprimono con le funzioni SQL.

## Step 8: Job batch con `spark-submit`
In produzione il codice non vive in un notebook ma in uno script inviato al cluster.
Lo script `jobs/wordcount_job.py` contiene lo stesso Word Count; il risultato viene salvato in `data/wordcount_out`.
Fermiamo prima la sessione del notebook (per liberare i core del cluster), eliminiamo l'output precedente e lanciamo `spark-submit`; il filtro `grep` toglie righe vuote e avvisi.

In [ ]:
# Libera i core del cluster: il job batch avrà una propria applicazione
spark.stop()
master = os.environ.get("SPARK_MASTER", "local[*]")
# Rimuove l'output precedente (Spark non sovrascrive una cartella esistente)
!rm -rf /home/jovyan/data/wordcount_out
# Invia lo script al cluster: argomenti = file di input e cartella di output
!spark-submit --master {master} --name WordCountBatch jobs/wordcount_job.py file:///home/jovyan/datasets/alice_in_wonderland.txt file:///home/jovyan/data/wordcount_out 2>&1 | grep -v -E "^\s*$|WARN"

### Step 8b: Verificare il risultato
Il job scrive un file `part-XXXXX` per partizione (qui una sola, per via di `coalesce(1)`), più il file `_SUCCESS`. Mostriamo la cartella e le prime 5 righe: le parole più frequenti.

In [ ]:
# Elenco dei file prodotti e prime righe (parola, conteggio)
!ls /home/jovyan/data/wordcount_out && head -5 /home/jovyan/data/wordcount_out/part-00000

### Punti chiave (spark-submit)
* `spark-submit` invia uno script come applicazione indipendente, con il proprio ciclo di vita.
* L'output è una cartella con un file per partizione e il marcatore `_SUCCESS`.

## Riepilogo

| Concetto | API o comando | A cosa serve |
|---|---|---|
| Lettura distribuita | `sc.textFile(path, minPartitions)` | divide il file in partizioni, una per task |
| Fase Map | `flatMap`, `map` | trasforma i record in coppie `(chiave, valore)` |
| Shuffle + Reduce | `reduceByKey` | aggrega i valori con la stessa chiave |
| Lineage | `rdd.toDebugString()` | mostra la catena di trasformazioni e gli stage |
| Ispezione partizioni | `glom()`, `mapPartitionsWithIndex` | vede come le chiavi si distribuiscono dopo lo shuffle |
| Pre-aggregazione | `reduceByKey` vs `groupByKey` | riduce i dati trasferiti nello shuffle (combiner) |
| Indice invertito | `distinct`, `groupByKey`, `lookup` | parola → elenco dei capitoli |
| API DataFrame | `explode`, `split`, `groupBy().count()`, `explain()` | stesso calcolo dichiarativo, ottimizzato da Catalyst |
| Confronto RDD e DataFrame | `F.sum(...).over(Window...)`, `collect_set` | stessi risultati e tempi a confronto, indice invertito dichiarativo |
| Job batch | `spark-submit --master ... script.py` | esegue uno script come applicazione sul cluster |